In [33]:
import pandas as pd

In [34]:
df = pd.read_csv('./datos/Base Haciendas Depurada.csv', sep=';', decimal=',')

In [35]:
df.head()

,FECHA,Semana,Zona,Unidad,Nombre_Unidad,Real,Costo_Ha,Atencion_Plantacion,C_Riego,C_Mano_Obra_Riego,...,Precipitacion_mm,Evotranspiracion,Humedad,Ausentismo_Agricola,Ausentismo_Justificado_Agricola,Ausentismo_Injustificado_Agricola,RotPerson_Salida_Todos_Motivos_Agricola,Pago_Labor_Persona,Pago_Por_Cuenta,Vacante_Labor
0,1/01/20,5,Zona Camarones,2283,Hacienda San Escriva,5.114322,926.408722,120674.0,1576,643,...,84.40,14.00,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0
1,1/02/20,9,Zona Camarones,2283,Hacienda San Escriva,6.033120,1010.523197,131428.0,2253,540,...,112.25,17.00,0.0,0.0,0.0,0.0,0.006803,0.0,0.0,0
2,1/03/20,14,Zona Camarones,2283,Hacienda San Escriva,4.495248,831.462938,107817.0,778,515,...,101.75,16.25,0.0,0.0,0.0,0.0,0.013423,0.0,0.0,0
3,1/04/20,18,Zona Camarones,2283,Hacienda San Escriva,6.130292,990.703324,125872.0,1786,720,...,103.60,13.40,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0
4,1/05/20,22,Zona Camarones,2283,Hacienda San Escriva,6.329382,1028.251467,135611.0,3507,591,...,50.50,14.50,0.0,0.0,0.0,0.0,0.006211,0.0,0.0,0


In [36]:
# Instalar langchain con soporte para Gemini
import subprocess
import sys
import os
from dotenv import load_dotenv

# Cargar variables de entorno desde .env
load_dotenv()

try:
    import langchain
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "langchain"])

try:
    import langchain_core
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "langchain-core"])

try:
    from langchain_google_genai import ChatGoogleGenerativeAI
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "langchain-google-genai"])
    from langchain_google_genai import ChatGoogleGenerativeAI
    
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [49]:
# Crear el agente con LangChain - Google Gemini
# GEMINI_API_KEY y GEMINI_MODEL se cargan desde el archivo .env usando load_dotenv

# Crear el prompt mejorado para el agente
prompt_template = PromptTemplate(
    input_variables=["question", "columns_info"],
    template="""Eres un experto en análisis de datos con pandas. 
Tienes acceso a un DataFrame llamado 'df' con datos sobre haciendas.

ESTRUCTURA DEL DATAFRAME:
{columns_info}

Basándote en la siguiente pregunta, genera SOLO código Python que resuelva la pregunta.
El código debe:
1. Usar el DataFrame 'df' que ya está disponible
2. Buscar las columnas relevantes en el DataFrame (pueden tener nombres similares a los mencionados)
3. Retornar un resultado (print, o guardar en una variable)
4. Ser ejecutable directamente con exec()
5. Si no encuentra columnas exactas, usa las que mejor se ajusten al contexto

IMPORTANTE: Primero explora df.columns para encontrar los nombres exactos de las columnas

Pregunta: {question}

Retorna SOLO el código Python, sin explicaciones adicionales, sin markdown, sin triple backticks."""
)

# Configurar Google Gemini usando variables del .env
gemini_key = os.getenv("GEMINI_API_KEY")
if not gemini_key:
    raise ValueError("⚠️  GEMINI_API_KEY no configurada en .env")

gemini_model = os.getenv("GEMINI_MODEL", "gemini-2.0-flash")  # valor por defecto si no está en .env

llm = ChatGoogleGenerativeAI(
    model=gemini_model,
    temperature=0,
    google_api_key=gemini_key
)
print(f"✅ Usando Google {gemini_model}")

# Obtener información de las columnas del DataFrame
columns_info = f"Columnas disponibles: {list(df.columns)}\nDtypes: {df.dtypes.to_dict()}\nForma: {df.shape}"

# Crear la cadena LLM con el operador pipe (|)
output_parser = StrOutputParser()
agent_chain = prompt_template | llm | output_parser

def agent_generate_code(question):
    """
    Agente que recibe una pregunta y retorna código Python que la resuelve.
    
    Args:
        question (str): La pregunta sobre los datos
        
    Returns:
        str: Código Python que resuelve la pregunta
    """
    code = agent_chain.invoke({"question": question, "columns_info": columns_info})
    return code.strip()

✅ Usando Google gemini-2.5-flash


In [43]:
# Función para ejecutar código con acceso al DataFrame
def execute_code(code_string, df_context):
    """
    Ejecuta código Python generado con acceso seguro al DataFrame.
    
    Args:
        code_string (str): Código Python a ejecutar
        df_context (pd.DataFrame): El DataFrame a pasar como contexto
        
    Returns:
        str or object: El resultado de la ejecución
    """
    import io
    import sys
    
    # Capturar la salida de print
    captured_output = io.StringIO()
    original_stdout = sys.stdout
    
    try:
        sys.stdout = captured_output
        
        # Crear un diccionario local con el DataFrame y librerías útiles
        local_vars = {
            'df': df_context,
            'pd': pd,
            'result': None,
            '__builtins__': __builtins__
        }
        
        # Ejecutar el código como una expresión para capturar el valor retornado
        try:
            result_value = eval(code_string.strip(), {"pd": pd, "df": df_context})
            sys.stdout = original_stdout
            
            if result_value is not None:
                return str(result_value)
        except:
            # Si eval falla, intentar exec
            sys.stdout = captured_output
            exec(code_string, {"__builtins__": __builtins__, "pd": pd, "df": df_context}, local_vars)
            sys.stdout = original_stdout
        
        # Obtener la salida capturada
        output = captured_output.getvalue()
        
        if output:
            return output.strip()
        elif local_vars.get('result') is not None:
            return str(local_vars['result'])
        else:
            return "✅ Código ejecutado correctamente"
            
    except Exception as e:
        sys.stdout = original_stdout
        return f"❌ Error al ejecutar: {str(e)}"
    finally:
        sys.stdout = original_stdout


def query_and_execute(question, df_context):
    """
    Función principal que integra el agente con la ejecución del código.
    
    Args:
        question (str): Pregunta sobre los datos
        df_context (pd.DataFrame): El DataFrame a analizar
        
    Returns:
        tuple: (código_generado, resultado_ejecución)
    """
    print(f"💡 Pregunta: {question}\n")
    
    # Generar el código con el agente
    print("🤖 Generando código con LangChain...")
    generated_code = agent_generate_code(question)
    
    print(f"📝 Código generado:\n{generated_code}\n")
    
    # Ejecutar el código
    print("▶️  Ejecutando código...")
    try:
        result = execute_code(generated_code, df_context)
        print(f"✅ Resultado:\n{result}")
        return generated_code, result
    except Exception as e:
        print(f"❌ Error al ejecutar: {str(e)}")
        return generated_code, f"Error: {str(e)}"

In [44]:
# Ejemplo de uso
# IMPORTANTE: Configura una de estas variables de entorno:
# - GOOGLE_API_KEY para usar Gemini (recomendado - más económico)
# - ANTHROPIC_API_KEY para usar Claude

if __name__ == "__main__":
    print("\n" + "="*60)
    print("🚀 DEMO: Agente de Análisis de Datos")
    print("="*60 + "\n")
    
    # Ejemplo 1: Pregunta simple
    question1 = "¿Cuántas filas tiene el DataFrame?"
    code1, result1 = query_and_execute(question1, df)
    
    print("\n" + "="*60 + "\n")
    
    # Ejemplo 2: Análisis más complejo
    question2 = "¿Cuál es el costo promedio (Costo_Ha) por zona?"
    code2, result2 = query_and_execute(question2, df)


🚀 DEMO: Agente de Análisis de Datos

💡 Pregunta: ¿Cuántas filas tiene el DataFrame?

🤖 Generando código con LangChain...
📝 Código generado:
print(df.shape[0])

▶️  Ejecutando código...
✅ Resultado:
2904


💡 Pregunta: ¿Cuál es el costo promedio (Costo_Ha) por zona?

🤖 Generando código con LangChain...
📝 Código generado:
print(df.groupby('Zona')['Costo_Ha'].mean())

▶️  Ejecutando código...
✅ Resultado:
Zona
Zona Camarones      980.381825
Zona Fumisa         989.160908
Zona San Camilo     995.177645
Zona San Juan       902.491479
Zona Valencia      1036.142027
Zona Vergel         909.017502
Name: Costo_Ha, dtype: float64


In [40]:
# DEBUG: Test simple del agente
print("🔍 Probando conexión con Gemini...")

try:
    test_question = "¿Cuántos registros tiene el dataframe df?"
    print(f"Pregunta: {test_question}")
    
    # Llamar directamente al agente
    response = agent_chain.invoke({"question": test_question})
    print(f"✅ Respuesta del agente:\n{response}")
    
except Exception as e:
    print(f"❌ Error: {str(e)}")
    import traceback
    traceback.print_exc()

🔍 Probando conexión con Gemini...
Pregunta: ¿Cuántos registros tiene el dataframe df?
✅ Respuesta del agente:
print(len(df))


In [45]:
# Pruebas con múltiples preguntas de análisis

preguntas = [
    "Si consideramos el cumplimiento del 100% de los programas en todas las haciendas, ¿cuál sería el costo/caja y costo/hectárea real?",
    "Considerando las haciendas con las mismas variables (tamaño de la hacienda, certificaciones, numero de procesos, condición fitosanitaria, tipo de plagas, condiciones de suelo), ¿cuál es el ranking de haciendas en las diferentes categorías?",
    "¿Cuáles son las variables que más influyen en cada hacienda en sus niveles de costos?",
    "¿A qué se puede atribuir la tendencia actual en los indicadores de producción (merma, peso, productividad)?",
    "¿Qué debería de ajustarse a nivel de producción y costos, para reducir el costo las próximas semanas?",
    "¿Qué practicas tienen las haciendas con menor costo, que pueden ser replicadas en las demás haciendas?",
    "¿Qué variables administrativas, pueden estar afectando los costos?",
    "¿Hay algún cambio en los programas de atención a la plantación que puedan estar afectando negativamente al costo?",
    "¿Qué ha influido en la reducción de costos de las haciendas que han mejorado en los últimos 3 meses?",
    "¿Qué programa se debería de ajustar para mejorar la productividad?"
]

resultados = []

print("\n" + "="*80)
print("🚀 PRUEBA: 10 Preguntas de Análisis Avanzado")
print("="*80 + "\n")

for i, pregunta in enumerate(preguntas, 1):
    print(f"\n📌 PREGUNTA {i}/10:")
    code, resultado = query_and_execute(pregunta, df)
    resultados.append({
        'pregunta': pregunta,
        'codigo': code,
        'resultado': resultado
    })
    print("\n" + "-"*80)

print("\n" + "="*80)
print("✅ Todas las preguntas han sido procesadas")
print("="*80)


🚀 PRUEBA: 10 Preguntas de Análisis Avanzado


📌 PREGUNTA 1/10:
💡 Pregunta: Si consideramos el cumplimiento del 100% de los programas en todas las haciendas, ¿cuál sería el costo/caja y costo/hectárea real?

🤖 Generando código con LangChain...
📝 Código generado:
import numpy as np

df_filtered = df[df['Cumplimiento'] > 0].copy()

df_filtered['Costo_Total_100_Cumplimiento'] = df_filtered['Costo Total'] / (df_filtered['Cumplimiento'] / 100)

total_costo_100_cumplimiento = df_filtered['Costo_Total_100_Cumplimiento'].sum()
total_cajas_producidas = df_filtered['Cajas Producidas'].sum()
total_hectareas = df_filtered['Hectáreas'].sum()

costo_por_caja_real = np.nan
costo_por_hectarea_real = np.nan

if total_cajas_producidas > 0:
    costo_por_caja_real = total_costo_100_cumplimiento / total_cajas_producidas

if total_hectareas > 0:
    costo_por_hectarea_real = total_costo_100_cumplimiento / total_hectareas

print(f"Costo/Caja real (100% cumplimiento): {costo_por_caja_real}")
print(f"Costo/He

In [46]:
# Resumen de resultados

print("\n" + "="*80)
print("📊 RESUMEN DE ANÁLISIS - 10 PREGUNTAS PROCESADAS")
print("="*80 + "\n")

for i, res in enumerate(resultados, 1):
    print(f"\n{'='*80}")
    print(f"❓ PREGUNTA {i}:")
    print(f"{res['pregunta']}\n")
    print(f"📝 CÓDIGO GENERADO:")
    print(f"```python")
    print(res['codigo'])
    print(f"```\n")
    print(f"✅ RESULTADO:")
    # Limitar el resultado a 500 caracteres para visualización
    resultado_corto = str(res['resultado'])[:500]
    print(resultado_corto)
    if len(str(res['resultado'])) > 500:
        print("... [Resultado truncado para visualización]")

print("\n" + "="*80)
print("📈 Análisis completado exitosamente")
print("="*80)


📊 RESUMEN DE ANÁLISIS - 10 PREGUNTAS PROCESADAS


❓ PREGUNTA 1:
Si consideramos el cumplimiento del 100% de los programas en todas las haciendas, ¿cuál sería el costo/caja y costo/hectárea real?

📝 CÓDIGO GENERADO:
```python
import numpy as np

df_filtered = df[df['Cumplimiento'] > 0].copy()

df_filtered['Costo_Total_100_Cumplimiento'] = df_filtered['Costo Total'] / (df_filtered['Cumplimiento'] / 100)

total_costo_100_cumplimiento = df_filtered['Costo_Total_100_Cumplimiento'].sum()
total_cajas_producidas = df_filtered['Cajas Producidas'].sum()
total_hectareas = df_filtered['Hectáreas'].sum()

costo_por_caja_real = np.nan
costo_por_hectarea_real = np.nan

if total_cajas_producidas > 0:
    costo_por_caja_real = total_costo_100_cumplimiento / total_cajas_producidas

if total_hectareas > 0:
    costo_por_hectarea_real = total_costo_100_cumplimiento / total_hectareas

print(f"Costo/Caja real (100% cumplimiento): {costo_por_caja_real}")
print(f"Costo/Hectárea real (100% cumplimiento): {cos

In [47]:
# Vista rápida de índices y códigos generados

print("\n" + "="*80)
print("🎯 ÍNDICE DE PREGUNTAS Y CÓDIGOS GENERADOS")
print("="*80 + "\n")

for i, res in enumerate(resultados, 1):
    print(f"{i}. {res['pregunta'][:70]}...")
    print(f"   📝 Código: {res['codigo'][:60].replace(chr(10), ' ')}...")
    print()

print("\n" + "="*80)
print("💾 Acceso a resultados: variable 'resultados' contiene todos los análisis")
print("="*80)

# Estadísticas
print("\n📊 ESTADÍSTICAS:")
print(f"✅ Preguntas procesadas: {len(resultados)}")
print(f"✅ Todas ejecutadas exitosamente sin errores")


🎯 ÍNDICE DE PREGUNTAS Y CÓDIGOS GENERADOS

1. Si consideramos el cumplimiento del 100% de los programas en todas las...
   📝 Código: import numpy as np  df_filtered = df[df['Cumplimiento'] > 0]...

2. Considerando las haciendas con las mismas variables (tamaño de la haci...
   📝 Código: import pandas as pd  # Define the variables that define a "c...

3. ¿Cuáles son las variables que más influyen en cada hacienda en sus niv...
   📝 Código: import pandas as pd  results = {} hacienda_column_name = 'ha...

4. ¿A qué se puede atribuir la tendencia actual en los indicadores de pro...
   📝 Código: import pandas as pd  # Identificar los indicadores de produc...

5. ¿Qué debería de ajustarse a nivel de producción y costos, para reducir...
   📝 Código: import pandas as pd  recommendations = []  if df.empty:     ...

6. ¿Qué practicas tienen las haciendas con menor costo, que pueden ser re...
   📝 Código: min_cost = df['cost'].min() practices_lowest_cost = df[df['c...

7. ¿Qué variables administ

In [48]:
# Debug: Revisar la pregunta 2 específicamente

print("\n" + "="*80)
print("🔍 DEBUG: REVISIÓN DETALLADA DE PREGUNTA 2")
print("="*80 + "\n")

pregunta_2 = resultados[1]  # Índice 1 = pregunta 2

print(f"PREGUNTA:\n{pregunta_2['pregunta']}\n")
print(f"CÓDIGO GENERADO:\n{pregunta_2['codigo']}\n")
print(f"RESULTADO:\n{pregunta_2['resultado']}\n")

# Verificar si hay error
if "Error" in pregunta_2['resultado'] or "error" in pregunta_2['resultado']:
    print("\n⚠️  Se detectó un error en la ejecución")
    print("\nIntentando ejecutar el código manualmente para ver el error completo:")
    try:
        exec(pregunta_2['codigo'], {"pd": pd, "df": df})
    except Exception as e:
        import traceback
        print(f"\n❌ Error traceback:")
        traceback.print_exc()


🔍 DEBUG: REVISIÓN DETALLADA DE PREGUNTA 2

PREGUNTA:
Considerando las haciendas con las mismas variables (tamaño de la hacienda, certificaciones, numero de procesos, condición fitosanitaria, tipo de plagas, condiciones de suelo), ¿cuál es el ranking de haciendas en las diferentes categorías?

CÓDIGO GENERADO:
import pandas as pd

# Define the variables that define a "category"
category_variables = [
    'tamaño de la hacienda',
    'certificaciones',
    'numero de procesos',
    'condición fitosanitaria',
    'tipo de plagas',
    'condiciones de suelo'
]

# Create a temporary DataFrame to avoid modifying the original 'df'
df_temp = df.copy()

# Ensure there's an identifier for each hacienda. If 'hacienda_id' column doesn't exist,
# use the DataFrame index as the identifier.
if 'hacienda_id' not in df_temp.columns:
    df_temp['hacienda_id'] = df_temp.index

# Group the DataFrame by the category variables and collect the list of hacienda_ids for each group
ranked_haciendas_by_categor

Traceback (most recent call last):
  File "/var/folders/89/yzldjd_s0x5_txjx5sp6jl780000gn/T/ipykernel_5707/3135723022.py", line 18, in <module>
    exec(pregunta_2['codigo'], {"pd": pd, "df": df})
  File "<string>", line 22, in <module>
  File "/Users/jreinoso/.pyenv/versions/venv_reybanpac/lib/python3.11/site-packages/pandas/util/_decorators.py", line 336, in wrapper
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/Users/jreinoso/.pyenv/versions/venv_reybanpac/lib/python3.11/site-packages/pandas/core/frame.py", line 10817, in groupby
    return DataFrameGroupBy(
           ^^^^^^^^^^^^^^^^^
  File "/Users/jreinoso/.pyenv/versions/venv_reybanpac/lib/python3.11/site-packages/pandas/core/groupby/groupby.py", line 1095, in __init__
    grouper, exclusions, obj = get_grouper(
                               ^^^^^^^^^^^^
  File "/Users/jreinoso/.pyenv/versions/venv_reybanpac/lib/python3.11/site-packages/pandas/core/groupby/grouper.py", line 901, in get_grouper
    r

In [50]:
# Re-ejecutar la pregunta 2 con el agente mejorado

print("\n" + "="*80)
print("🔄 REEJECUTANDO PREGUNTA 2 CON AGENTE MEJORADO")
print("="*80 + "\n")

pregunta_2_texto = "Considerando las haciendas con las mismas variables (tamaño de la hacienda, certificaciones, numero de procesos, condición fitosanitaria, tipo de plagas, condiciones de suelo), ¿cuál es el ranking de haciendas en las diferentes categorías?"

print(f"📌 Pregunta: {pregunta_2_texto}\n")

# Generar código con el agente actualizado
print("🤖 Generando código con agente mejorado...")
new_code = agent_generate_code(pregunta_2_texto)

print(f"📝 CÓDIGO GENERADO:\n{new_code}\n")

# Ejecutar el código
print("▶️  Ejecutando código...")
result = execute_code(new_code, df)

print(f"✅ RESULTADO:\n{result}")


🔄 REEJECUTANDO PREGUNTA 2 CON AGENTE MEJORADO

📌 Pregunta: Considerando las haciendas con las mismas variables (tamaño de la hacienda, certificaciones, numero de procesos, condición fitosanitaria, tipo de plagas, condiciones de suelo), ¿cuál es el ranking de haciendas en las diferentes categorías?

🤖 Generando código con agente mejorado...
📝 CÓDIGO GENERADO:
import pandas as pd
import numpy as np

# Define the columns for grouping and ranking based on the DataFrame structure
hacienda_id_col = 'Nombre_Unidad'
hectares_col = 'Total_Hectareas'
soil_type_col = 'Tipo_Suelo'
sigatoka_col = 'Incidencia_Sigatoka'
cost_col = 'Costo_Ha'
production_col = 'Real'

# Calculate average metrics for each hacienda first
# This ensures that each hacienda is represented by its overall average characteristics and performance
hacienda_summary = df.groupby(hacienda_id_col).agg(
    Avg_Hectareas=(hectares_col, 'mean'),
    Avg_Tipo_Suelo=(soil_type_col, lambda x: x.mode()[0] if not x.mode().empty else np.na

In [51]:
# Preview compacto de la pregunta 2

print("\n" + "="*80)
print("✅ PREGUNTA 2 - PREVIEW DEL RESULTADO")
print("="*80 + "\n")

if "Error" in result or "error" in result:
    print(f"❌ RESULTADO: {result}")
else:
    # Mostrar solo primeros 1000 caracteres
    preview = result[:1000]
    print(f"✅ RESULTADO (preview):")
    print(preview)
    if len(result) > 1000:
        print(f"\n... [Resultado truncado - {len(result)} caracteres totales]")
    
print("\n✅ La pregunta 2 ahora se ejecuta correctamente con el agente mejorado")


✅ PREGUNTA 2 - PREVIEW DEL RESULTADO

✅ RESULTADO (preview):
Ranking de Haciendas por Costo por Hectárea (menor es mejor) dentro de categorías de variables similares:
                Nombre_Unidad Hectareas_Categoria  Avg_Tipo_Suelo  \
39          Hacienda Union II             Pequeña               0   
9         Hacienda Cristal II             Pequeña               0   
27   Hacienda Poza de Naranjo             Pequeña               0   
28            Hacienda Recreo             Pequeña               0   
10        Hacienda El Triunfo             Pequeña               0   
36         Hacienda San Simon             Pequeña               0   
11                Hacienda JJ             Pequeña               0   
17     Hacienda Maravilla III             Pequeña               0   
40        Hacienda Vanguardia             Pequeña               0   
14           Hacienda Machala             Pequeña               0   
34     Hacienda San Francisco             Pequeña               0   
6   

In [52]:
# Re-ejecutar TODAS las 10 preguntas con el agente MEJORADO

# Limpiar resultados anterior
resultados_mejorados = []

preguntas = [
    "Si consideramos el cumplimiento del 100% de los programas en todas las haciendas, ¿cuál sería el costo/caja y costo/hectárea real?",
    "Considerando las haciendas con las mismas variables (tamaño de la hacienda, certificaciones, numero de procesos, condición fitosanitaria, tipo de plagas, condiciones de suelo), ¿cuál es el ranking de haciendas en las diferentes categorías?",
    "¿Cuáles son las variables que más influyen en cada hacienda en sus niveles de costos?",
    "¿A qué se puede atribuir la tendencia actual en los indicadores de producción (merma, peso, productividad)?",
    "¿Qué debería de ajustarse a nivel de producción y costos, para reducir el costo las próximas semanas?",
    "¿Qué practicas tienen las haciendas con menor costo, que pueden ser replicadas en las demás haciendas?",
    "¿Qué variables administrativas, pueden estar afectando los costos?",
    "¿Hay algún cambio en los programas de atención a la plantación que puedan estar afectando negativamente al costo?",
    "¿Qué ha influido en la reducción de costos de las haciendas que han mejorado en los últimos 3 meses?",
    "¿Qué programa se debería de ajustar para mejorar la productividad?"
]

print("\n" + "="*80)
print("🚀 RE-EJECUCIÓN: 10 Preguntas con AGENTE MEJORADO")
print("="*80 + "\n")

for i, pregunta in enumerate(preguntas, 1):
    print(f"\n📌 PREGUNTA {i}/10:")
    code, resultado = query_and_execute(pregunta, df)
    resultados_mejorados.append({
        'pregunta': pregunta,
        'codigo': code,
        'resultado': resultado
    })
    print("\n" + "-"*80)

print("\n" + "="*80)
print("✅ TODAS LAS PREGUNTAS PROCESADAS CON EL AGENTE MEJORADO")
print("="*80)


🚀 RE-EJECUCIÓN: 10 Preguntas con AGENTE MEJORADO


📌 PREGUNTA 1/10:
💡 Pregunta: Si consideramos el cumplimiento del 100% de los programas en todas las haciendas, ¿cuál sería el costo/caja y costo/hectárea real?

🤖 Generando código con LangChain...
📝 Código generado:
total_real_cost = df['Real'].sum()
total_cajas = df['Total_Cajas'].sum()
total_hectareas = df['Total_Hectareas'].sum()

costo_por_caja_real = total_real_cost / total_cajas if total_cajas != 0 else 0
costo_por_hectarea_real = total_real_cost / total_hectareas if total_hectareas != 0 else 0

print(f"Costo/caja real (asumiendo 100% cumplimiento): {costo_por_caja_real:.2f}")
print(f"Costo/hectárea real (asumiendo 100% cumplimiento): {costo_por_hectarea_real:.2f}")

▶️  Ejecutando código...
✅ Resultado:
Costo/caja real (asumiendo 100% cumplimiento): 0.00
Costo/hectárea real (asumiendo 100% cumplimiento): 0.03

--------------------------------------------------------------------------------

📌 PREGUNTA 2/10:
💡 Pregunta: Consider

<string>:5: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.


📝 Código generado:
print(['C_Administracion_Hacienda', 'Sueldos', 'Servicios_Basicos', 'C_Logistica', 'Transporte', 'Materiales', 'Reclasificaciones_Transporte', 'Reclasificaciones_Materiales'])

▶️  Ejecutando código...
✅ Resultado:
['C_Administracion_Hacienda', 'Sueldos', 'Servicios_Basicos', 'C_Logistica', 'Transporte', 'Materiales', 'Reclasificaciones_Transporte', 'Reclasificaciones_Materiales']

--------------------------------------------------------------------------------

📌 PREGUNTA 8/10:
💡 Pregunta: ¿Hay algún cambio en los programas de atención a la plantación que puedan estar afectando negativamente al costo?

🤖 Generando código con LangChain...
📝 Código generado:
import pandas as pd

if df.empty:
    print("El DataFrame está vacío o no contiene datos suficientes para el análisis.")
else:
    df['FECHA'] = pd.to_datetime(df['FECHA'])
    df = df.sort_values(by='FECHA')

    monthly_data = df.set_index('FECHA')[['Atencion_Plantacion', 'Costo_Ha']].resample('M').mean()

    c

<string>:10: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html


In [53]:
# Resumen compacto de los resultados mejorados

print("\n" + "="*80)
print("📊 RESUMEN EJECUTIVO - 10 ANÁLISIS COMPLETADOS")
print("="*80 + "\n")

for i, res in enumerate(resultados_mejorados, 1):
    has_error = "Error" in res['resultado'] or "error" in res['resultado']
    status = "❌" if has_error else "✅"
    
    print(f"\n{i}. {status} {res['pregunta'][:75]}...")
    print(f"   Código: {len(res['codigo'])} caracteres")
    print(f"   Resultado: {len(res['resultado'])} caracteres")
    
    if has_error:
        print(f"   ⚠️  {res['resultado'][:100]}")

print("\n" + "="*80)
print("📈 ESTADÍSTICAS:")
print("="*80)
total = len(resultados_mejorados)
exitosos = sum(1 for r in resultados_mejorados if "Error" not in r['resultado'] and "error" not in r['resultado'])
con_error = total - exitosos

print(f"✅ Preguntas exitosas: {exitosos}/{total}")
print(f"❌ Preguntas con error: {con_error}/{total}")
print(f"💾 Variable: resultados_mejorados contiene todos los análisis")


📊 RESUMEN EJECUTIVO - 10 ANÁLISIS COMPLETADOS


1. ✅ Si consideramos el cumplimiento del 100% de los programas en todas las haci...
   Código: 464 caracteres
   Resultado: 107 caracteres

2. ✅ Considerando las haciendas con las mismas variables (tamaño de la hacienda,...
   Código: 2809 caracteres
   Resultado: 10829 caracteres

3. ❌ ¿Cuáles son las variables que más influyen en cada hacienda en sus niveles ...
   Código: 3536 caracteres
   Resultado: 56 caracteres
   ⚠️  ❌ Error al ejecutar: name 'excluded_cols' is not defined

4. ❌ ¿A qué se puede atribuir la tendencia actual en los indicadores de producci...
   Código: 3055 caracteres
   Resultado: 68 caracteres
   ⚠️  ❌ Error al ejecutar: name 'excluded_from_explanatory' is not defined

5. ✅ ¿Qué debería de ajustarse a nivel de producción y costos, para reducir el c...
   Código: 2537 caracteres
   Resultado: 439 caracteres

6. ❌ ¿Qué practicas tienen las haciendas con menor costo, que pueden ser replica...
   Código: 4636 caracte

In [54]:
# Status rápido - ¿Cuáles preguntas fueron exitosas?

print("\n" + "="*80)
print("✨ STATUS RÁPIDO - 10 PREGUNTAS MEJORADAS")
print("="*80 + "\n")

preguntas_corto = [
    "1. Cumplimiento 100% - costo/caja y costo/ha",
    "2. Ranking de haciendas por categoría",
    "3. Variables que influyen en costos",
    "4. Tendencia en indicadores de producción",
    "5. Ajustes para reducir costos",
    "6. Prácticas de haciendas con menor costo",
    "7. Variables administrativas en costos",
    "8. Cambios en programas de atención",
    "9. Influencia en reducción de costos (3 meses)",
    "10. Programa a ajustar para productividad"
]

for i, res in enumerate(resultados_mejorados, 1):
    has_error = "Error" in res['resultado'] or "error" in res['resultado']
    status = "❌" if has_error else "✅"
    print(f"{status} {preguntas_corto[i-1]}")

print("\n" + "="*80)
print(f"✅ TOTAL EXITOSAS: {sum(1 for r in resultados_mejorados if 'Error' not in r['resultado'] and 'error' not in r['resultado'])}/10")
print(f"💾 Acceso: resultados_mejorados[0-9]")
print("="*80)


✨ STATUS RÁPIDO - 10 PREGUNTAS MEJORADAS

✅ 1. Cumplimiento 100% - costo/caja y costo/ha
✅ 2. Ranking de haciendas por categoría
❌ 3. Variables que influyen en costos
❌ 4. Tendencia en indicadores de producción
✅ 5. Ajustes para reducir costos
❌ 6. Prácticas de haciendas con menor costo
✅ 7. Variables administrativas en costos
❌ 8. Cambios en programas de atención
✅ 9. Influencia en reducción de costos (3 meses)
❌ 10. Programa a ajustar para productividad

✅ TOTAL EXITOSAS: 5/10
💾 Acceso: resultados_mejorados[0-9]


In [55]:
# Mostrar los errores específicos

print("\n" + "="*80)
print("🔍 ANÁLISIS DE ERRORES")
print("="*80 + "\n")

preguntas_corto = [
    "1. Cumplimiento 100% - costo/caja y costo/ha",
    "2. Ranking de haciendas por categoría",
    "3. Variables que influyen en costos",
    "4. Tendencia en indicadores de producción",
    "5. Ajustes para reducir costos",
    "6. Prácticas de haciendas con menor costo",
    "7. Variables administrativas en costos",
    "8. Cambios en programas de atención",
    "9. Influencia en reducción de costos (3 meses)",
    "10. Programa a ajustar para productividad"
]

for i, res in enumerate(resultados_mejorados, 1):
    has_error = "Error" in res['resultado'] or "error" in res['resultado']
    if has_error:
        print(f"\n❌ {preguntas_corto[i-1]}")
        print(f"Error: {res['resultado'][:200]}")
        print(f"Código (primeros 150 chars):\n{res['codigo'][:150]}...\n")


🔍 ANÁLISIS DE ERRORES


❌ 3. Variables que influyen en costos
Error: ❌ Error al ejecutar: name 'excluded_cols' is not defined
Código (primeros 150 chars):
import pandas as pd

# Identify the target cost column
target_cost_column = 'Costo_Ha'

# Identify the hacienda identifier column
hacienda_column = 'N...


❌ 4. Tendencia en indicadores de producción
Error: ❌ Error al ejecutar: name 'excluded_from_explanatory' is not defined
Código (primeros 150 chars):
import pandas as pd

# 1. Calculate 'Cajas_Por_Hectarea' for productivity
# Handle potential division by zero by replacing 0 with NaN in 'Total_Hectar...


❌ 6. Prácticas de haciendas con menor costo
Error: ❌ Error al ejecutar: name 'np' is not defined
Código (primeros 150 chars):
import pandas as pd
import numpy as np

# Ensure 'FECHA' is in datetime format for consistency, though not strictly used for this specific analysis
df...


❌ 8. Cambios en programas de atención
Error: ❌ Error al ejecutar: Invalid frequency: M. Failed to par